# Project 08: The Viterbi algorithm for hidden Markov models

This notebook implements the **Viterbi algorithm** for hidden Markov models (HMMs)

Given a sequence of observations, Viterbi finds the **single sequence of hidden states** with the highest joint probability with the observations (the *decoding* problem; Rabiner, 1989). We follow the Viterbi formula as given in the course primer, A primer on HMMs (Figure 3, notation from Rabiner, 1989):, in **log units** to avoid underflow.

**Sections**
1. Imports and settings
2. The `HMM` class (Stage 1: set up and check the model; Stages 2–4: Viterbi)
3. Demonstration on the example data
4. Tests

## 1. Imports and settings
Only the Python standard library is used: `math` for logs, and `itertools` for the brute-force test.
`NEG_INF` stands for log(0), an impossible event. `TOLERANCE` is how far a row of probabilities may be from 1 because of rounding.

In [1]:
import math

NEG_INF = float("-inf")
TOLERANCE = 1e-6

# itertools.product lists every possible path, for the brute-force test
import itertools

## 2. The `HMM` class

Functions 1–4 are **Stage 1** (set up and check the model). Function 5 is **Viterbi** (Stages 2–4), following the formula in the course primer (Figure 3, notation from Rabiner, 1989):

| Step | Formula |
|---|---|
| Start | `delta_1(i) = pi_i * b_i(o_1),  psi_1(i) = 0,  1 <= i <= N` |
| Move forward one position | `delta_t(j) = max_i( delta_(t-1)(i) * a_ij ) * b_j(o_t),  2 <= t <= T` |
| | `psi_t(j) = argmax_i( delta_(t-1)(i) * a_ij ),  2 <= t <= T` |
| Finish | `P* = max_i( delta_T(i) ),  q*_T = argmax_i( delta_T(i) )` |
| Trace back | `q*_t = psi_(t+1)( q*_(t+1) ),  t = T-1, T-2, ..., 1` |

Our code uses logs, so `*` becomes `+`.

| README function | In the class |
|---|---|
| 1. Create the model | `__init__()` |
| 2. Safe log | `safe_log()` |
| 3. Validate | `_validate()` |
| 4. Convert to logs | `_convert_to_logs()` |
| 5. Viterbi decoding | `viterbi()` |

In [2]:
class HMM:
    # PART 1: build the model

    def __init__(self, init_probs, trans_probs, emit_probs):
        # Save the original tables
        self.init_probs = init_probs
        self.trans_probs = trans_probs
        self.emit_probs = emit_probs

        # Get states and symbols from the data (never hard-coded)
        # example: states = ['I', 'G'], symbols = ['A', 'C', 'G', 'T']
        self.states = list(init_probs.keys())
        if not self.states:
            raise ValueError("init_probs has no states")
        first_state = self.states[0]
        self.symbols = list(emit_probs.get(first_state, {}).keys())

        # Check the tables, then convert everything to logs once
        self._validate()
        self._convert_to_logs()

    @staticmethod
    def safe_log(prob):
        """Natural log, except log(0) = -inf (an impossible event)."""
        if prob == 0:
            return NEG_INF
        return math.log(prob)

    def _validate(self):
        states = set(self.states)
        symbols = set(self.symbols)

        # 1. Keys must match across the three tables
        if set(self.trans_probs.keys()) != states:
            raise ValueError("trans_probs states don't match init_probs states")
        if set(self.emit_probs.keys()) != states:
            raise ValueError("emit_probs states don't match init_probs states")
        for s in self.states:
            if set(self.trans_probs[s].keys()) != states:
                raise ValueError(f"trans_probs['{s}'] doesn't list every state")
            if set(self.emit_probs[s].keys()) != symbols:
                raise ValueError(f"emit_probs['{s}'] doesn't list every symbol")

        # 2. Every value must be a real probability
        all_values = list(self.init_probs.values())
        for s in self.states:
            all_values += list(self.trans_probs[s].values())
            all_values += list(self.emit_probs[s].values())
        for p in all_values:
            if not 0 <= p <= 1:
                raise ValueError(f"probability {p} is not between 0 and 1")

        # 3. Every row must add up to 1 (with a little rounding room)
        if abs(sum(self.init_probs.values()) - 1) > TOLERANCE:
            raise ValueError("init_probs don't add up to 1")
        for s in self.states:
            if abs(sum(self.trans_probs[s].values()) - 1) > TOLERANCE:
                raise ValueError(f"trans_probs['{s}'] don't add up to 1")
            if abs(sum(self.emit_probs[s].values()) - 1) > TOLERANCE:
                raise ValueError(f"emit_probs['{s}'] don't add up to 1")

    def _convert_to_logs(self):
        # example: log_init['G'] = ln(0.8) = -0.2231
        self.log_init = {s: self.safe_log(self.init_probs[s]) for s in self.states}
        # example: log_trans['G']['G'] = ln(0.9) = -0.1054
        self.log_trans = {
            i: {j: self.safe_log(self.trans_probs[i][j]) for j in self.states}
            for i in self.states
        }
        # example: log_emit['I']['C'] = ln(0.4) = -0.9163
        self.log_emit = {
            s: {x: self.safe_log(self.emit_probs[s][x]) for x in self.symbols}
            for s in self.states
        }

    # PART 2: Viterbi

    def viterbi(self, sequence):
        """
        Return (path, best_log_prob): the most likely hidden-state path for
        the sequence, and the natural log of that path's probability.
        """
        # Check the input
        if len(sequence) == 0:
            raise ValueError("empty sequence")
        for symbol in sequence:
            if symbol not in self.log_emit[self.states[0]]:
                raise ValueError(f"unknown symbol '{symbol}'")

        n = len(sequence)
        # score[t][state] = best log score for reaching this state at column t
        # arrow[t][state] = which state that best score came from
        score = [dict() for _ in range(n)]
        arrow = [dict() for _ in range(n)]

        # Column 1 (index 0): log initial + log emission
        # example: score[0]['G'] = ln(0.8) + ln(0.2) = -1.8326
        first = sequence[0]
        for s in self.states:
            score[0][s] = self.log_init[s] + self.log_emit[s][first]
            arrow[0][s] = None

        # Columns 2 to the end: try every way in, keep the best
        for t in range(1, n):
            symbol = sequence[t]
            for state_now in self.states:
                best_road = NEG_INF
                best_from = None
                for state_before in self.states:
                    road = score[t - 1][state_before] + self.log_trans[state_before][state_now]
                    if road > best_road:
                        best_road = road
                        best_from = state_before
                score[t][state_now] = best_road + self.log_emit[state_now][symbol]
                arrow[t][state_now] = best_from

        # Best ending: biggest score in the last column
        best_last_state = max(self.states, key=lambda s: score[n - 1][s])
        best_log_prob = score[n - 1][best_last_state]
        if best_log_prob == NEG_INF:
            raise ValueError("no possible path for this sequence")

        # Trace back: stand on a box, read its arrow, jump left
        path = []
        standing_on = best_last_state
        for t in range(n - 1, -1, -1):
            path.append(standing_on)
            standing_on = arrow[t][standing_on]
        path.reverse()

        return path, best_log_prob

## 3. Demonstration on the example data

The four data structures from the project description. **I** = CpG island state, **G** = genome background state (not the letter G).

In [3]:
# Example observation sequence
obs = "GGCACTGAA"

# Example initial probabilities (probability of starting in each state)
init_probs = {"I": 0.2, "G": 0.8}

# Example transition probabilities (probability of moving from one state to another)
trans_probs = {"I": {"I": 0.7, "G": 0.3},
               "G": {"I": 0.1, "G": 0.9}}

# Example emission probabilities (probability of observing a symbol in a given state)
emit_probs = {"I": {"A": 0.1, "C": 0.4, "G": 0.4, "T": 0.1},
              "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}}

In [4]:
# build the model (this also runs _validate and _convert_to_logs)
hmm = HMM(init_probs, trans_probs, emit_probs)
print("states :", hmm.states)
print("symbols:", hmm.symbols)

# decode the example sequence
path, log_prob = hmm.viterbi(obs)
print()
print("observation :", obs)
print("best path   :", "".join(path))
print("log prob    :", round(log_prob, 4))
print("probability :", f"{math.exp(log_prob):.4g}")

states : ['I', 'G']
symbols: ['A', 'C', 'G', 'T']

observation : GGCACTGAA
best path   : GGGGGGGGG
log prob    : -13.9291
probability : 8.926e-07


## 4. Tests
A small helper (the first cell below) prints PASS or FAIL for each test, so a failure is easy to spot. All tests are in the second cell, grouped by what they check. The purpose of each test is listed in section 6 of the README.
1. Hand-checked tests (worked out by hand first)
2. The example data
3. Input errors
4. A different model (the course primer's coverage track)
5. Brute force (Viterbi must match the best of every possible path)
6. Three states
7. Zero probabilities
8. Long sequence
9. Underflow (why we use logs)

In [5]:
# compare two log probabilities with a small tolerance (floating point is not exact)
def close(a, b, tol=1e-4):
    return abs(a - b) <= tol


# print PASS/FAIL for one test
def check(name, ok):
    print(("PASS  " if ok else "FAIL  ") + name)


# run a call that SHOULD raise ValueError; PASS if it does
def check_raises(name, func):
    try:
        func()
    except ValueError as e:
        print(f"PASS  {name}  ->  ValueError: {e}")
        return
    print(f"FAIL  {name}  (no error was raised)")

In [6]:
# ---------- 1. Hand-checked tests (worked out by hand first) ----------
# one symbol: start in G, 0.8 x 0.2 = 0.16
p, lp = hmm.viterbi("G")
check("one symbol:      viterbi('G')   -> G, prob 0.16", p == ["G"] and close(lp, math.log(0.16)))
# two symbols: stay in G, 0.16 x 0.9 x 0.2 = 0.0288
p, lp = hmm.viterbi("GG")
check("two symbols:     viterbi('GG')  -> G G, prob 0.0288", p == ["G", "G"] and close(lp, math.log(0.0288)))
# three symbols: the island state wins at the letter C
p, lp = hmm.viterbi("GGC")
check("three symbols:   viterbi('GGC') -> I I I, prob 0.006272", p == ["I", "I", "I"] and close(lp, math.log(0.006272)))

# ---------- 2. The example data ----------
p, lp = hmm.viterbi("GGCACTGAA")
check("class example:   viterbi('GGCACTGAA') -> all G, log prob -13.9291", p == ["G"] * 9 and close(lp, -13.9291))

# ---------- 3. Input errors: fail loudly instead of returning a meaningless answer ----------
# a symbol no state can emit
check_raises("unknown symbol:  viterbi('GGNA')", lambda: hmm.viterbi("GGNA"))
# an empty sequence has no positions, so there is no path
check_raises("empty sequence:  viterbi('')", lambda: hmm.viterbi(""))
# a broken table: emissions of I add up to 0.9
check_raises("broken table:    emissions of I add up to 0.9",
             lambda: HMM(init_probs, trans_probs,
                         {"I": {"A": 0.1, "C": 0.4, "G": 0.3, "T": 0.1},
                          "G": {"A": 0.3, "C": 0.2, "G": 0.2, "T": 0.3}}))

# ---------- 4. A different model: the course primer's coverage track ----------
primer = HMM({"T": 0.2, "U": 0.8},
             {"T": {"T": 0.9, "U": 0.1}, "U": {"T": 0.05, "U": 0.95}},
             {"T": {"none": 0.05, "few": 0.15, "many": 0.80},
              "U": {"none": 0.70, "few": 0.28, "many": 0.02}})
p, lp = primer.viterbi(["none", "few", "many", "many", "none", "many", "none"])
check("different model: primer track -> U U T T T T U, prob about 9.7e-06",
      p == ["U", "U", "T", "T", "T", "T", "U"] and abs(math.exp(lp) - 9.7e-6) < 0.1e-6)

# ---------- 5. Brute force: Viterbi must match the best of every possible path ----------
# log joint probability of ONE path, straight from the definition
def path_log_prob(m, path, seq):
    total = m.log_init[path[0]] + m.log_emit[path[0]][seq[0]]
    for t in range(1, len(seq)):
        total += m.log_trans[path[t - 1]][path[t]] + m.log_emit[path[t]][seq[t]]
    return total

# try every possible path and keep the best one
def brute_force(m, seq):
    best_path, best_lp = None, NEG_INF
    for path in itertools.product(m.states, repeat=len(seq)):
        lp = path_log_prob(m, path, seq)
        if lp > best_lp:
            best_path, best_lp = list(path), lp
    return best_path, best_lp

for seq in ["GGCACTGAA", "CGCGCGCG", "ATATCGCGCG", "ACGT", "CCCCGGGGA"]:
    vp, vlp = hmm.viterbi(seq)
    bp, blp = brute_force(hmm, seq)
    check(f"brute force:     {seq:12s} -> {''.join(vp)}", vp == bp and close(vlp, blp, 1e-9))

# ---------- 6. Three states (no assumption about the number of states) ----------
three = HMM({"A": 0.5, "B": 0.3, "C": 0.2},
            {"A": {"A": 0.6, "B": 0.3, "C": 0.1},
             "B": {"A": 0.2, "B": 0.5, "C": 0.3},
             "C": {"A": 0.1, "B": 0.2, "C": 0.7}},
            {"A": {"x": 0.7, "y": 0.2, "z": 0.1},
             "B": {"x": 0.1, "y": 0.8, "z": 0.1},
             "C": {"x": 0.2, "y": 0.1, "z": 0.7}})
for seq in ["xyz", "xxyyzz", "zzzyyx"]:
    vp, vlp = three.viterbi(seq)
    bp, blp = brute_force(three, seq)
    check(f"three states:    {seq:12s} -> {''.join(vp)}", vp == bp and close(vlp, blp, 1e-9))

# ---------- 7. Zero probabilities ----------
# initial probability of I is 0: the path must never start in I
p, lp = HMM({"I": 0.0, "G": 1.0}, trans_probs, emit_probs).viterbi("CGCG")
check(f"initial prob 0:  path starts in {p[0]}, never in I", p[0] == "G")
# every path impossible: I only makes A, G only makes C, and no state can switch
locked = HMM({"I": 0.5, "G": 0.5},
             {"I": {"I": 1.0, "G": 0.0}, "G": {"I": 0.0, "G": 1.0}},
             {"I": {"A": 1.0, "C": 0.0}, "G": {"A": 0.0, "C": 1.0}})
check_raises("impossible path: viterbi('AC')", lambda: locked.viterbi("AC"))

# ---------- 8. Long sequence: logs keep the score a real number ----------
p, lp = hmm.viterbi("GGCACTGAA" * 200)
check(f"long sequence:   1,800 symbols -> log prob {lp:.4f}", len(p) == 1800 and lp != NEG_INF and not math.isnan(lp))

# ---------- 9. Underflow: why we use logs ----------
# keep multiplying by 0.25 until the computer rounds the result to 0
value, steps = 1.0, 0
while value > 0:
    value *= 0.25
    steps += 1
check(f"underflow:       0.25 multiplied {steps} times becomes 0.0", value == 0.0)
# the all-G path on 900 letters: plain probabilities (multiplying) versus our logs
long_seq = obs * 100
plain = init_probs["G"] * emit_probs["G"][long_seq[0]]
for letter in long_seq[1:]:
    plain *= trans_probs["G"]["G"] * emit_probs["G"][letter]
p, lp = hmm.viterbi(long_seq)
check(f"underflow:       900 letters -> plain probability {plain}, log prob {lp:.4f}",
      plain == 0.0 and lp != NEG_INF)

PASS  one symbol:      viterbi('G')   -> G, prob 0.16
PASS  two symbols:     viterbi('GG')  -> G G, prob 0.0288
PASS  three symbols:   viterbi('GGC') -> I I I, prob 0.006272
PASS  class example:   viterbi('GGCACTGAA') -> all G, log prob -13.9291
PASS  unknown symbol:  viterbi('GGNA')  ->  ValueError: unknown symbol 'N'
PASS  empty sequence:  viterbi('')  ->  ValueError: empty sequence
PASS  broken table:    emissions of I add up to 0.9  ->  ValueError: emit_probs['I'] don't add up to 1
PASS  different model: primer track -> U U T T T T U, prob about 9.7e-06
PASS  brute force:     GGCACTGAA    -> GGGGGGGGG
PASS  brute force:     CGCGCGCG     -> IIIIIIII
PASS  brute force:     ATATCGCGCG   -> GGGGIIIIII
PASS  brute force:     ACGT         -> GGGG
PASS  brute force:     CCCCGGGGA    -> IIIIIIIIG
PASS  three states:    xyz          -> ABC
PASS  three states:    xxyyzz       -> AABBCC
PASS  three states:    zzzyyx       -> CCCBBA
PASS  initial prob 0:  path starts in G, never in I
PASS  imp

All 21 tests pass. What these results show, and their limitations, are discussed in the README (Successes and Struggles).